# 📈 MNQ 14:50 Strategy — Google Colab Runner

This notebook lets you run the full **MNQ 14:50 Backtesting & Prop Firm Optimization System** directly on Google Colab — no local Python setup required.

### Steps
1. **Run Cell 1** — Clone repo & install dependencies  
2. **Run Cell 2** — Upload your 1-minute bar CSV data file  
3. **Run the module cells** you want (Core, Validation, Volatility, Prop Firm)  
4. **View charts & results** inline or download from the output folder

---
### CSV Format Required
| Column | Format | Description |
|--------|--------|-------------|
| `timestamp ET` | `MM/DD/YYYY HH:MM` | Bar timestamp in Eastern Time |
| `open` | float | Open price |
| `high` | float | High price |
| `low` | float | Low price |
| `close` | float | Close price |
| `volume` | int | Bar volume |
| `Vwap_RTH` | float (optional) | VWAP during regular trading hours |

1-minute bars, multi-year history.

## ⚙️ Cell 1 — Setup: Clone Repository & Install Dependencies

In [ ]:
import subprocess, sys, os

# ── Clone the strategy repository ────────────────────────────────────────────
REPO_URL  = "https://github.com/Keshav12kai/mnq-1450-strategy.git"
REPO_DIR  = "/content/mnq-1450-strategy"

if not os.path.isdir(REPO_DIR):
    print("Cloning repository …")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("✓ Repository cloned")
else:
    print("Repository already present — pulling latest …")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    print("✓ Repository up to date")

# Add repo to Python path so we can import its modules
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ── Install Python dependencies ───────────────────────────────────────────────
print("\nInstalling dependencies …")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "pandas", "numpy", "matplotlib", "scipy", "scikit-learn"],
    check=True
)
print("✓ Dependencies installed")

# ── Switch working directory into the repo ───────────────────────────────────
os.chdir(REPO_DIR)
print(f"\nWorking directory: {os.getcwd()}")
print("\n✅ Setup complete — proceed to Cell 2 to upload your CSV.")

## 📂 Cell 2 — Upload CSV Data File

Run this cell and click **Choose Files** to upload your 1-minute bar CSV.  
The uploaded file will be saved to `/content/data.csv`.

In [ ]:
from google.colab import files as colab_files
import shutil, os

print("Please upload your 1-minute bar CSV file …")
uploaded = colab_files.upload()

if not uploaded:
    raise RuntimeError("No file was uploaded. Please re-run this cell and upload your CSV.")

# Move the uploaded file to a fixed location so later cells always find it
uploaded_name = list(uploaded.keys())[0]
CSV_PATH = "/content/data.csv"
shutil.copy(f"/content/{uploaded_name}", CSV_PATH)

import pandas as pd
preview = pd.read_csv(CSV_PATH, nrows=3)
print(f"\n✅ File saved to: {CSV_PATH}")
print(f"   Columns : {list(preview.columns)}")
print(f"   Preview :")
print(preview.to_string(index=False))

## 🔧 Cell 3 — Configuration (Optional)

Adjust capital, point value, and output folder here before running any modules.

In [ ]:
import os

# ── User-configurable parameters ─────────────────────────────────────────────
CAPITAL      = 10_000   # Initial capital in dollars
POINT_VALUE  = 2.0      # MNQ = $2/point;  NQ = $20/point
OUTPUT_DIR   = "/content/mnq_output"   # Where charts & CSVs are saved

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration:")
print(f"  CSV path    : {CSV_PATH}")
print(f"  Capital     : ${CAPITAL:,.0f}")
print(f"  Point value : ${POINT_VALUE}/pt")
print(f"  Output dir  : {OUTPUT_DIR}")
print("\n✅ Configuration set — run the module cells below.")

## 📊 Cell 4 — Core Strategy Backtester

Runs the main backtest (Long-only, Short-only, Both directions) and Buy & Hold benchmark.  
Prints a side-by-side stats table and saves TradingView-style charts.

In [ ]:
import importlib, time
import matplotlib
matplotlib.use("Agg")   # keep Agg for file saves …
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image
import glob, os

# Reload modules in case of re-run
import core_strategy as cs
importlib.reload(cs)

print("=" * 60)
print("  MODULE: Core Strategy Backtester")
print("=" * 60)
t0 = time.time()

results, trades, df = cs.run(
    CSV_PATH,
    initial_capital=CAPITAL,
    point_value=POINT_VALUE,
    output_dir=OUTPUT_DIR,
)

print(f"\n✓ Core module finished in {time.time()-t0:.1f}s")

# ── Display all saved charts inline ──────────────────────────────────────────
print("\n── Charts ──")
chart_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.png")))
if chart_files:
    for path in chart_files:
        print(f"  {os.path.basename(path)}")
        display(Image(filename=path))
else:
    print("  (no PNG charts found in output directory)")

## 🧪 Cell 5 — Advanced Validation

Runs the full statistical validation suite:  
Monte Carlo (10,000 paths), Walk-Forward Analysis, Pattern Detection, t-test / Bootstrap CI.

In [ ]:
import importlib, time, glob, os
import numpy as np
from IPython.display import display, Image
import config

import advanced_validation as av
importlib.reload(av)

# Make sure core data is loaded (run Cell 4 first, or load here)
if 'trades' not in dir() or trades is None:
    import core_strategy as cs
    importlib.reload(cs)
    df     = cs.load_data(CSV_PATH)
    trades = cs.extract_trades(df)

print("=" * 60)
print("  MODULE: Advanced Validation")
print("=" * 60)
t0 = time.time()

filtered = trades[~trades["skip"]].copy()
filtered["pnl"] = np.where(
    filtered["direction"] == "LONG",
    filtered["pnl_long_pts"]  * POINT_VALUE,
    filtered["pnl_short_pts"] * POINT_VALUE,
)

av.run(filtered, pnl_col="pnl", initial_capital=CAPITAL, output_dir=OUTPUT_DIR)
print(f"\n✓ Validation module finished in {time.time()-t0:.1f}s")

# ── Display new charts ────────────────────────────────────────────────────────
print("\n── Charts ──")
chart_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.png")))
for path in chart_files:
    print(f"  {os.path.basename(path)}")
    display(Image(filename=path))

## 🤖 Cell 6 — Volatility Predictor & Position Sizer

Trains 5 ML models (EWMA, P75, Feature Scaling, Gradient Boosting, **Ensemble**) to predict the full 14:50–14:59 window range and sizes positions to avoid daily-loss-limit breaches.

In [ ]:
import importlib, time, glob, os
from IPython.display import display, Image

import volatility_predictor as vp
importlib.reload(vp)

if 'trades' not in dir() or trades is None or 'df' not in dir() or df is None:
    import core_strategy as cs
    importlib.reload(cs)
    df     = cs.load_data(CSV_PATH)
    trades = cs.extract_trades(df)

print("=" * 60)
print("  MODULE: Volatility Predictor")
print("=" * 60)
t0 = time.time()

vp.run(df, trades, output_dir=OUTPUT_DIR)
print(f"\n✓ Volatility predictor finished in {time.time()-t0:.1f}s")

# ── Display new charts ────────────────────────────────────────────────────────
print("\n── Charts ──")
chart_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.png")))
for path in chart_files:
    print(f"  {os.path.basename(path)}")
    display(Image(filename=path))

## 🏦 Cell 7 — Prop Firm Challenge Optimizer

Sweeps 1–50 contracts across 25+ prop firms (Topstep, Apex, MFF, TradeDay, …) and ranks them by pass rate, ROI, and expected cost to get funded.

In [ ]:
import importlib, time, glob, os
from IPython.display import display, Image

import propfirm_optimizer as pfo
importlib.reload(pfo)

if 'trades' not in dir() or trades is None:
    import core_strategy as cs
    importlib.reload(cs)
    df     = cs.load_data(CSV_PATH)
    trades = cs.extract_trades(df)

print("=" * 60)
print("  MODULE: Prop Firm Optimizer")
print("=" * 60)
t0 = time.time()

pfo.run(trades, output_dir=OUTPUT_DIR)
print(f"\n✓ Prop firm optimizer finished in {time.time()-t0:.1f}s")

# ── Display new charts ────────────────────────────────────────────────────────
print("\n── Charts ──")
chart_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.png")))
for path in chart_files:
    print(f"  {os.path.basename(path)}")
    display(Image(filename=path))

## 💾 Cell 8 — Download All Results

Zips every chart (PNG) and trade log (CSV) produced by the above modules and downloads the archive to your computer.

In [ ]:
import zipfile, os, glob
from google.colab import files as colab_files

ZIP_PATH = "/content/mnq_strategy_results.zip"

output_files = (
    glob.glob(os.path.join(OUTPUT_DIR, "*.png")) +
    glob.glob(os.path.join(OUTPUT_DIR, "*.csv"))
)

if not output_files:
    print("No output files found yet — run at least one module cell first.")
else:
    with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in output_files:
            zf.write(f, arcname=os.path.basename(f))
    print(f"Zipped {len(output_files)} file(s) → {ZIP_PATH}")
    colab_files.download(ZIP_PATH)
    print("✅ Download started.")